# Surface-enumeration Python API

This tutorial configures and enumerates vacancy and substitution orderings in surface slabs using the public Python API. It separates structure loading, layer analysis, composition setup, external enumeration, result inspection, symmetric-slab handling, and optional file export so that each step can be inspected independently.

In [ ]:
from pathlib import Path
from shutil import which

from surface_pd.core import EnumerationSlab, SurfaceEnumerator

enumlib_available = which("enum.x") is not None and any(
    which(name) is not None
    for name in ("makestr.x", "makeStr.x", "makeStr.py")
)

The notebook expects to be started from the `examples/` directory. 

## 1. Load the slab structure

The VASP file `POSCAR_LI_100.vasp` contains an asymmetric nine-layer Li(100) slab with its vacuum region along direction 2 (the third lattice vector), and selective-dynamics flags that identify its fixed bottom region and relaxed surface. We will substitute Na for some of the Li in the surface layer. 

In [ ]:
structure_path = Path(
    "enumeration-examples/structure/electrode/POSCAR_Li_100.vasp"
)

Next, we read and configure the slab in one step. The `EnumerationSlab.from_file()` factory transparently uses pymatgen to read its supported structure formats. Here `direction=2` selects the third lattice direction, which contains the broken periodicity and vacuum region; the lattice vector need not be perpendicular to the surface plane. `enumerated_species` selects Li, and `num_enumerated_layers={"Li": 1}` selects the outermost Li-bearing layer. Layer counts are independent for each selected species: for example, `{"Li": 1, "O": 2}` would select the outermost Li-bearing layer and two outermost O-bearing layers. Sites whose Cartesian coordinates normal to the in-plane lattice span no more than `layer_tolerance_angstrom` are assigned to the same layer.

With `symmetric=False`, only the top surface is enumerated. Setting `symmetric=True` instead selects corresponding layers on both surfaces and requires a compatible symmetric slab. Our Li(100) slab model is asymmetric, and its bottom layers are fixed, so we have to disable symmetry. Printing the configured slab retains pymatgen's structure summary and appends these surface-enumeration settings.

In [ ]:
slab = EnumerationSlab.from_file(
    structure_path,
    direction=2,
    layer_tolerance_angstrom=0.5,
    enumerated_species=["Li"],
    num_enumerated_layers={"Li": 1},
    symmetric=False,
)

print(slab)

## 2. Inspect layers and selected sites

Calling `EnumerationSlab.analyze()` returns one immutable snapshot of the slab's current state. Its layer records are ordered from the bottom to the top surface and contain a mean Cartesian coordinate in angstroms, the original site indices, and species counts. The coordinates are measured normal to the two in-plane lattice vectors, so they remain physically meaningful for slanted cells. The same snapshot reports exactly which original sites may change during enumeration, which sites are fixed by selective-dynamics flags, and the Cartesian bounds of those fixed sites. For this asymmetric slab, the fixed region is at the bottom rather than at the center. Enumeration selection and selective-dynamics constraints are independent concepts.

In [ ]:
analysis = slab.analyze()
print(analysis)

## 3. Define the ordered surface configurations

Next, we configure the enumeration. The replacement mapping below requests 50% Li to be replaced with Na. Introducing vacancies instead of other chemical species is also possible by a mapping such as `{"Li": {"Li": 0.5}}`, which would remove half of the enumerated Li atoms. 

The enumeration is done using the enumlib library and requires its binaries to be set up correctly. For each requested composition, enumlib finds symmetrically distinct orderings over the allowed surface cells. Since the surface unit cell of the original slab model only contains a single Li atom, an area multiplier of at least two is needed to realize the requested replacement (`min_cell_size=2`). Larger in-plane cell multipliers permit more orderings but can greatly increase the computational effort. We will limit the enumeration to a multiple of 4 (`max_cell_size=4`).

At this stage, no enumeration is performed yet. We are only configuring the enumeration settings.

In [ ]:
enumerator = SurfaceEnumerator(
    replacements={"Li": {"Li": 0.5, "Na": 0.5}},
    min_cell_size=2,
    max_cell_size=4,
)

## 4. Run the external enumeration when requested

The enumeration is performed with `SurfaceEnumerator.apply_enumeration`, which delegates candidate generation to enumlib through pymatgen and returns finalized `EnumerationSlab` objects. Note that the `max_structures` argument limits the ranked raw candidates requested from enumlib; surface filtering happens afterward, so fewer finalized structures may be returned. 

Each returned slab has immutable `enumeration_metadata` describing its in-plane transformation, area multiplier, zero-based raw candidate rank, and symmetry mode. Results retain raw candidate order. Surface-pd does not apply an additional structure-matching or deduplication pass after finalization. Enumlib already removes symmetry-equivalent raw candidates, while a second tolerance-dependent comparison could merge physically distinct surface orderings and become expensive for large result sets. Applications that need a different equivalence policy can explicitly group selected results with pymatgen's `StructureMatcher`.

The execution of the following cell is opt-in because it can take a little while to complete. Also note that enumlib is separately installed. To run the cell, install `enum.x` and `makestr.x` (or a supported `makeStr` variant), make them discoverable on `PATH`, set `RUN_ENUMLIB = True`, and rerun this cell.

In [ ]:
enumerated_slabs = []
RUN_ENUMLIB = False
# RUN_ENUMLIB = True

if RUN_ENUMLIB and enumlib_available:
    enumerated_slabs = enumerator.apply_enumeration(
        slab, max_structures=20
    )
    print(f"enumlib returned {len(enumerated_slabs)} surface slab(s)")
    for index, result in enumerate(enumerated_slabs):
        metadata = result.enumeration_metadata
        print(index, result.formula, metadata.area_multiplier)
elif RUN_ENUMLIB:
    print("enumlib was requested but its executables were not found")
else:
    print("enumlib execution skipped by default; deterministic setup complete")

When enumeration is enabled, the following cell inspects the top layer of the last finalized slab. A lightweight two-dimensional comparison is being designed separately; this tutorial keeps the structural inspection textual and dependency-free.

In [ ]:
if RUN_ENUMLIB and enumlib_available:
    top_layer = enumerated_slabs[-1].analyze().layers[-1]
    print("top layer coordinate (angstrom):", top_layer.coordinate)
    print("top layer site indices:", top_layer.site_indices)

## 5. Symmetric enumeration on both surfaces

Modifying the surface layer of a periodic slab model may introduce a dipole moment and lead to artificial results in electronic structure calculations without dipole correction. This is especially significant for non-metallic materials. To avoid such dipole moments, surface-pd supports symmetric enumeration, where the top and bottom surfaces of a slab model are modified in equivalent and symmetric fashion. 

Symmetric enumeration requires more than an inversion-symmetric lattice: the selective-dynamics flags must identify relaxed regions on both surfaces around a fixed central region. The one-sided Li(100) slab above is intentionally asymmetric and not suitable for a symmetric enumeration. Instead, we will use a Li$_2$O(110) slab for a symmetric enumeration example. The outermost Li layer and two outermost O layers of the Li$_2$O slab are selected independently on both surfaces.

In [ ]:
symmetric_structure_path = Path(
    "enumeration-examples/structure/electrode/POSCAR_Li2O_110.vasp"
)
symmetric_slab = EnumerationSlab.from_file(
    symmetric_structure_path,
    direction=2,
    layer_tolerance_angstrom=0.5,
    enumerated_species=["Li", "O"],
    num_enumerated_layers={"Li": 1, "O": 2},
    symmetric=True,
)

print(
    "selected symmetric sites:",
    dict(symmetric_slab.analyze().enumerated_site_indices),
)

For this example, we will keep the Li sites fully occupied while only 75% of the selected O sites remain occupied. Surface-pd first enumerates one surface, restores complete selective-dynamics metadata, mirrors the ordering onto the second surface, and returns only finalized inversion-symmetric slabs.

In [ ]:
symmetric_enumerator = SurfaceEnumerator(
    replacements={"Li": {"Li": 1.0}, "O": {"O": 0.75}},
    min_cell_size=1,
    max_cell_size=2,
)

In [ ]:
symmetric_results = []
RUN_SYMMETRIC_ENUMLIB = False
# RUN_SYMMETRIC_ENUMLIB = True

if RUN_SYMMETRIC_ENUMLIB and enumlib_available:
    symmetric_results = symmetric_enumerator.apply_enumeration(
        symmetric_slab, max_structures=20
    )
    print(f"enumlib returned {len(symmetric_results)} symmetric slab(s)")
    for index, result in enumerate(symmetric_results):
        print(
            index, result.formula, result.lattice.abc,
            result.has_inversion_symmetry(),
        )
elif RUN_SYMMETRIC_ENUMLIB:
    print(
        "symmetric enumlib execution requested but executables were not found"
    )
else:
    print("symmetric enumlib execution skipped by default")

## 6. Export finalized slabs as VASP POSCAR files

Every finalized result is a pymatgen-compatible `EnumerationSlab`, so it can be written directly in VASP POSCAR format. Export is opt-in to keep ordinary notebook execution free of generated files. Set `WRITE_POSCARS = True` only after running at least one enumlib section. Files are written below `generated-poscars/`, relative to the `examples/` working directory, and their names include the in-plane area multiplier and raw candidate rank. Selective-dynamics flags are retained by pymatgen.

In [ ]:
WRITE_POSCARS = False
# WRITE_POSCARS = True

result_groups = (
    ("Li100", enumerated_slabs),
    ("Li2O110-symmetric", symmetric_results),
)

if WRITE_POSCARS:
    if not any(structures for _, structures in result_groups):
        raise RuntimeError("run an enumlib section before exporting")
    output_directory = Path("generated-poscars")
    output_directory.mkdir(exist_ok=True)
    written_paths = []
    for label, structures in result_groups:
        for result in structures:
            metadata = result.enumeration_metadata
            filename = (
                f"POSCAR_{label}_area{metadata.area_multiplier}_"
                f"rank{metadata.raw_candidate_rank:04d}.vasp"
            )
            path = output_directory / filename
            result.to(filename=path, fmt="poscar")
            written_paths.append(path)
    print(f"wrote {len(written_paths)} POSCAR file(s)")
else:
    print("POSCAR export skipped by default")